# Sanity 01 — the data and the pinned benchmark

Mirrors `scripts/01_generate_data.py`. Two artifacts come out of stage 01:
`raw/<scale>/transactions.parquet` (the normalized 24.4M-row TabFormer table + `splits.json`
temporal cutoffs) and `raw/<scale>/benchmark.parquet` — **the pinned NVIDIA-protocol eval rows**
(1M balanced train + 100k stratified val/test) that every headline number is measured on.
Run this on a workspace with `/mnt/user_storage` mounted.

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))   # template root (src/)
BASE = os.environ.get("FM_BASE", "/mnt/user_storage/transaction-fm-v2")
SCALE = "full"                               # flip to "xl" / "xxl" to inspect those runs

In [ ]:
import pyarrow.dataset as pads

raw = pads.dataset(f"{BASE}/raw/{SCALE}/transactions.parquet", format="parquet")
print(f"{raw.count_rows():,} transactions, columns: {raw.schema.names}")
splits = json.load(open(f"{BASE}/raw/{SCALE}/splits.json"))
print("temporal cutoffs:", {k: v for k, v in splits.items() if "end" in str(k)})

In [ ]:
# One card's life, in time order — this is exactly what one model window sees.
cols = ["card_id", "timestamp", "amount", "merchant_id", "mcc", "channel", "is_fraud"]
sample = raw.head(200_000, columns=cols).to_pandas()
card = sample["card_id"].value_counts().index[0]
hist = sample[sample.card_id == card].sort_values("timestamp")
print(f"card {card}: {len(hist)} txns in this sample, {hist.is_fraud.sum()} fraud")
hist.head(10)

In [ ]:
# The pinned benchmark: NVIDIA's notebook-01 sampling, persisted once, never regenerated casually.
bench = pd.read_parquet(f"{BASE}/raw/{SCALE}/benchmark.parquet")
print(bench.groupby("split").agg(rows=("_target", "size"), fraud_rate=("_target", "mean")))
bench.head(3)

**What to check:** train ≈ 1M rows at ~2.5% fraud (TabFormer only has ~25k train-period frauds,
so the 10% balanced target caps there); val/test = 100k at natural ~0.1%. The `_ts`/`card_id`
columns are the join keys embeddings are matched back on (plus amount-in-cents).

**Also on disk: `raw/<scale>/benchmark_fulltest.parquet`** — identical train/val rows (same seeded protocol code) but test = the ENTIRE test period (2,442,795 rows, 2,724 frauds). This is the eval behind the tight-CI table: same protocol, sampling variance removed. Rule of the house: numbers from the 100k benchmark and the fulltest benchmark are **never comparable to each other** — the 100k table compares against NVIDIA published points, the fulltest table compares our models against each other.